# Bronze Layer

## Step -1  Define Schema as STRING

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("store_location", StringType(), True),
    StructField("order_status", StringType(), True)
])

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("loyalty_points", StringType(), True),
    StructField("status", StringType(), True)
])

## Step -2 Read Raw as CSV

In [0]:
orders_bronze_df = spark.read.format('csv').option("header", "true").schema(orders_schema).load("/Volumes/retail_project/raw_data/raw_file/retail_orders_messy.csv")

customers_bronze_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(customer_schema) \
    .load("/Volumes/retail_project/raw_data/raw_file/retail_customers_messy.csv")


In [0]:
orders_bronze_df.display()
customers_bronze_df.display()

## Step - 3 Add Ingestion metadata

In [0]:
from pyspark.sql.functions import current_timestamp


orders_bronze_df = orders_bronze_df.withColumn('ingest_timestamp', current_timestamp())
customers_bronze_df = customers_bronze_df.withColumn('ingest_timestamp',current_timestamp())

## Step -4 Save AS Delta Table in Bronze

In [0]:
orders_bronze_df.write.format("delta").mode("overwrite").saveAsTable('retail_project.bronze.bronze_orders')
customers_bronze_df.write.format("delta").mode("overwrite").saveAsTable('retail_project.bronze.bronze_customers')


In [0]:
# Retail Voulume
# │
# ├── raw_file/
# │   ├── retail_orders/
# │   └── retail_customers/
# │
# ├── bronze/
# │   ├── bronze_orders
# │   └── bronze_customers

*************************************************************************

### Silver Layer Transformation 

![](/Volumes/ytdemo/data/demodata/image.png)

**🎯 TODAY’S OBJECTIVE**


In this lecture, we will clean the retail orders and customer dataset by:

- Fixing date formats

- Standardizing category values

- Removing duplicates

- Handling null values

- Converting discount from "5%" to 0.05 and "10%" to 0.1

- Removing negative quantities

- Fixing casing issues

- Casting unit_price to decimal

- Validating emails

- Recalculating total_amount

- **-- and more**

### STEP 1: Load Bronze Orders

In [0]:
bronze_order_df = spark.read.table("retail_project.bronze.bronze_orders")

display(bronze_order_df)

### 👨‍🏫 STEP 2: Fix Date Formats

In [0]:
from pyspark.sql.functions import try_to_date, col, coalesce

silver_df = bronze_order_df.withColumn(
    'order_date',
    coalesce(
        try_to_date(col('order_date'),'yyyy-MM-dd'),
        try_to_date(col('order_date'),'dd-MM-yyyy'),
        try_to_date(col('order_date'),'yyyy/MM/dd'),
        try_to_date(col('order_date'),'dd/MM/yyyy')
    )
)
silver_df.display()

### 👨‍🏫 STEP 3: Standardize Category Values

In [0]:
silver_df.select('category').distinct().display()

In [0]:
from pyspark.sql.functions import upper, trim,when

silver_df = silver_df.withColumn(
    "category",
    upper(trim(col("category")))
)


silver_df = silver_df.withColumn(
        'category',when(col('category')=='ELECTRONIC','ELECTRONICS').
        when(col('category')=='HOME AND LIVING','HOME & LIVING').
        when(col('category')=='FASHON','FASHION').
        otherwise(col('category')))

## STEP 4: Remove Duplicate Orders

In [0]:
from pyspark.sql.functions import count ,countDistinct

silver_df.select(count('order_id'),countDistinct('order_id')).display()

In [0]:
silver_df = silver_df.dropDuplicates(['order_id'])

silver_df.select(count('order_id'),countDistinct('order_id')).display()

### 👨‍🏫 STEP 5: Handle Null Values discount

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn('discount',
                                 when(col('discount').isNull(),'0').when(col('discount')=='ten','0.1').otherwise(col('discount'))
                                 
                                 )

In [0]:
silver_df.display()

### 👨‍🏫 STEP 6: Convert Discount from "5%" to Decimal

some rows contain:

5% 

we need 0.05

In [0]:
from pyspark.sql.functions import regexp_replace

silver_df = silver_df.withColumn(
    "discount",
    when(col("discount").like("5%"), 
           (regexp_replace("discount", "5%", "0.05"))
    ).when(col("discount").like("10%"), 
           (regexp_replace("discount", "10%", "0.1"))
    ).otherwise(col("discount"))
)

silver_df.display()

### 👨‍🏫 STEP 7: Remove Negative Quantity

In [0]:
silver_df = silver_df.filter(col("quantity")>=0)

In [0]:
silver_df.display()

### 👨‍🏫 STEP 8: Cast unit_price to Decimal

In [0]:
from pyspark.sql.types import DecimalType

silver_df = silver_df.withColumn(
    "unit_price",
    col("unit_price").try_cast(DecimalType(10,2))
)
silver_df.display()

### 👨‍🏫 STEP 9: Recalculate Total Amount

In [0]:
silver_df = silver_df.withColumn(
    "total_amount",
    col("quantity").cast("int") * col("unit_price") * (1 - col("discount").cast("double"))
)

In [0]:
silver_df.display()

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.silver.silver_orders")

### Retail Customers – Silver Layer Cleaning | Databricks End-to-End Project

### 👨‍🏫 STEP 1: Read Bronze Table

In [0]:
customers_bronze_df = spark.table("retail_project.bronze.bronze_customers")

display(customers_bronze_df)

### 👨‍🏫 STEP 2: Trim Spaces

In [0]:
from pyspark.sql.functions import trim, col

customers_df = customers_bronze_df.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    trim(col("email")).alias("email"),
    trim(col("phone")).alias("phone"),
    trim(col("gender")).alias("gender"),
    trim(col("date_of_birth")).alias("date_of_birth"),
    trim(col("city")).alias("city"),
    trim(col("state")).alias("state"),
    trim(col("registration_date")).alias("registration_date"),
    trim(col("loyalty_points")).alias("loyalty_points"),
    trim(col("status")).alias("status"),
    col("ingest_timestamp")
)

display(customers_df)

### 👨‍🏫 STEP 3: Remove Duplicate Customers

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("customer_id") \
                    .orderBy(desc("ingest_timestamp"))

customers_df = customers_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

customers_df = customers_df.filter("row_num = 1") \
                           .drop("row_num")

customers_df.display()

### 👨‍🏫 STEP 4: Fix and Standardize Status

In [0]:
from pyspark.sql.functions import upper, when

customers_df = customers_df.withColumn(
    "status",
    upper(col("status"))
)

customers_df = customers_df.withColumn(
    "status",
    when(col("status").isin("ACTIVE", "INACTIVE"), col("status"))
    .otherwise("UNKNOWN")
)

customers_df.display()

### 👨‍🏫 STEP 5: Convert loyalty_points to Integer

In [0]:
from pyspark.sql.functions import regexp_replace

customers_df = customers_df.withColumn(
    "loyalty_points",
    regexp_replace(col("loyalty_points"), "[^0-9]", "")
)

customers_df = customers_df.withColumn(
    "loyalty_points",
    col("loyalty_points").try_cast("int")
)

In [0]:
customers_df.display()

### 👨‍🏫 STEP 6: Validate Email Format

In [0]:
customers_df = customers_df.withColumn(
    "email_valid",
    col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$")
)

# Example: meenadas@email.com - valid, karansingh@email - invalid

# customers_df.select("email", "email_valid").display()

customers_df = customers_df.filter(col("email_valid") == True)

### 👨‍🏫 STEP 7: Fix Date Formats

In [0]:
from pyspark.sql.functions import to_date, coalesce

customers_df = customers_df.withColumn(
    "dob_parsed",
    coalesce(
            try_to_date(col('date_of_birth'),'yyyy-MM-dd'),
            try_to_date(col('date_of_birth'),'dd-MM-yyyy'),
            try_to_date(col('date_of_birth'),'yyyy/MM/dd'),
            try_to_date(col('date_of_birth'),'dd/MM/yyyy')
        )
)

customers_df = customers_df.withColumn(
    "registration_parsed",
    coalesce(
            try_to_date(col('registration_date'),'yyyy-MM-dd'),
            try_to_date(col('registration_date'),'dd-MM-yyyy'),
            try_to_date(col('registration_date'),'yyyy/MM/dd'),
            try_to_date(col('registration_date'),'dd/MM/yyyy')
        )
)

customers_df.display()

In [0]:
customers_df = customers_df.drop("date_of_birth", "registration_date") \
                           .withColumnRenamed("dob_parsed", "date_of_birth") \
                           .withColumnRenamed("registration_parsed", "registration_date")

In [0]:
customers_df.display()

### 👨‍🏫 STEP 8: Standardize Gender

In [0]:
customers_df = customers_df.withColumn(
    "gender",
    upper(col("gender"))
)

customers_df = customers_df.withColumn(
    "gender",
    when(col("gender") == "MALE", "M")
    .when(col("gender") == "FEMALE", "F")
    .otherwise("U")
)

In [0]:
customers_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.silver.silver_customers")